In [ ]:
import requests
import pandas as pd
import time

url_base = "https://resultados.tse.jus.br"

ambiente = "oficial"
ciclo = "ele2026"
codigo_eleicao = 6257
codigo_eleicao_formatado = f"{codigo_eleicao:06d}"
codigo_cargo = "0001"  # Presidente


url_municipios = (
    f"{url_base}/"
    f"{ambiente}/"
    f"{ciclo}/"
    f"{codigo_eleicao}/"
    f"config/"
    f"mun-e{codigo_eleicao_formatado}-cm.json"
)

resposta = requests.get(url_municipios, timeout=30)

resposta.raise_for_status()

dados_municipios = resposta.json()

municipios = []

for estado in dados_municipios.get("abr", []):
    uf = estado["cd"].lower()
    nome_estado = estado["ds"]

    if uf == "zz":
        continue

    for municipio in estado.get("mu", []):
        codigo_tse = str(municipio["cd"]).zfill(5)
        codigo_ibge = str(municipio.get("cdi", "")).zfill(5)
        nome_municipio = municipio["nm"]
        municipios.append(
            {
                "uf": uf,
                "estado": nome_estado,
                "codigo_tse": codigo_tse,
                "codigo_ibge": codigo_ibge,
                "municipio": nome_municipio,
            }
        )

print("Quantidade de municípios encontrados:", len(municipios))

resultados = []

for indice, municipio in enumerate(municipios, start=1):
    uf = municipio["uf"]
    estado = municipio["estado"]
    codigo_tse = municipio["codigo_tse"]
    codigo_ibge = municipio["codigo_ibge"]
    nome_municipio = municipio["municipio"]

    url_resultado = (
        f"{url_base}/"
        f"{ambiente}/"
        f"{ciclo}/"
        f"{codigo_eleicao}/"
        f"dados/{uf}/"
        f"{uf}{codigo_tse}"
        f"-c{codigo_cargo}"
        f"-e{codigo_eleicao_formatado}"
        f"-u.json"
    )

    try:
        resposta = requests.get(url_resultado, timeout=30)

        if resposta.status_code == 404:
            print(f"Arquivo não encontrado: {nome_municipio} - {uf.upper()}")
            continue

        resposta.raise_for_status()
        dados = resposta.json()

        votos = dados.get("v", {})
        eleitores = dados.get("e", {})
        secoes = dados.get("s", {})
        total_votos = votos.get("tv", 0)
        votos_validos = votos.get("vv", 0)
        votos_brancos = votos.get("vb", 0)
        total_votos_nulos = votos.get("tvn", 0)
        votos_nulos = votos.get("vn", 0)
        votos_nulos_tecnicos = votos.get("vnt", 0)
        votos_anulados = votos.get("van", 0)
        votos_anulados_sub_judice = votos.get("vansj", 0)
        votos_sem_candidato = votos.get("vscv", 0)
        votos_secoes_anuladas = votos.get("vsan", 0)

        votos_invalidos = (
            votos_nulos
            + votos_anulados
            + votos_anulados_sub_judice
            + votos_secoes_anuladas
        )

        votos_nao_validos = votos_brancos + votos_invalidos

        eleitorado = eleitores.get("te", 0)
        comparecimento = eleitores.get("c", 0)
        abstencoes = eleitores.get("a", 0)
        total_secoes = int(secoes.get("ts", 0) or 0)
        secoes_totalizadas = int(secoes.get("st", 0) or 0)
        secoes_nao_totalizadas = secoes.get("snt", 0)
        percentual_secoes_totalizadas = (
            secoes_totalizadas / total_secoes * 100 if total_secoes else 0
        )

        for cargo in dados.get("carg", []):
            for agrupamento in cargo.get("agr", []):
                for partido in agrupamento.get("par", []):
                    for candidato in partido.get("cand", []):
                        resultados.append(
                            {
                                "estado": estado,
                                "uf": uf.upper(),
                                "municipio": nome_municipio,
                                "codigo_tse": codigo_tse,
                                "codigo_ibge": codigo_ibge,
                                "numero_candidato": candidato.get("n"),
                                "nome_candidato": candidato.get("nm"),
                                "nome_urna": candidato.get("nmu"),
                                "partido": partido.get("sg"),
                                "destinacao_voto": candidato.get("dvt"),
                                "situacao": candidato.get("st"),
                                "eleito": candidato.get("e"),
                                "votos_candidato": candidato.get("vap", 0),
                                "percentual_candidato": candidato.get("pvap"),
                                "percentual_candidato_preciso": candidato.get("pvapn"),
                                "total_votos": total_votos,
                                "votos_validos": votos_validos,
                                "votos_brancos": votos_brancos,
                                "total_votos_nulos": total_votos_nulos,
                                "votos_nulos": votos_nulos,
                                "votos_nulos_tecnicos": votos_nulos_tecnicos,
                                "votos_anulados": votos_anulados,
                                "votos_anulados_sub_judice": votos_anulados_sub_judice,
                                "votos_invalidos": votos_invalidos,
                                "votos_nao_validos": votos_nao_validos,
                                "votos_sem_candidato": votos_sem_candidato,
                                "votos_secoes_anuladas": votos_secoes_anuladas,
                                "eleitorado": eleitorado,
                                "comparecimento": comparecimento,
                                "abstencoes": abstencoes,
                                "total_secoes": total_secoes,
                                "secoes_totalizadas": secoes_totalizadas,
                                "secoes_nao_totalizadas": secoes_nao_totalizadas,
                                "percentual_secoes_totalizadas": percentual_secoes_totalizadas,
                                "andamento": dados.get("and"),
                                "totalizacao_final": dados.get("tf"),
                                "data_totalizacao": dados.get("dt"),
                                "hora_totalizacao": dados.get("ht"),
                                "data_geracao": dados.get("hg"),
                            }
                        )

        if indice % 100 == 0:
            print(f"{indice} de {len(municipios)} municípios processados")

        time.sleep(0.03)

    except requests.RequestException as erro:
        print(f"Erro em {nome_municipio} - {uf.upper()}: {erro}")


df_eleicao = pd.DataFrame(resultados)
df_eleicao = df_eleicao.sort_values(
    by=["uf", "municipio", "votos_candidato"],
    ascending=[True, True, False],
).reset_index(drop=True)

print()
print("Quantidade de linhas:", len(df_eleicao))
print()
print("Quantidade de municípios:", df_eleicao["municipio"].nunique())
print()
print("Quantidade de candidatos:", df_eleicao["nome_urna"].nunique())

df_eleicao

Quantidade de municípios encontrados: 5571
100 de 5571 municípios processados
200 de 5571 municípios processados
300 de 5571 municípios processados
